In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch version: 2.13.0+cpu
CUDA available: False
Running on CPU


In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [3]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.14.1
Uninstalling transformers-5.14.1:
  Successfully uninstalled transformers-5.14.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)

   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transfor

In [4]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

# Download NLTK tokenizer
nltk.download("punkt")

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", device)

c:\Projects\Text-Summarizer-project_01\Text-Summarizer-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.13.0+cpu
Device: cpu


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rupan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [6]:
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 4154.18it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
import os

print("Current folder:")
print(os.getcwd())

print("\nZIP path:")
zip_path = os.path.abspath("../summarizer-data.zip")
print(zip_path)

print("\nZIP exists:")
print(os.path.exists(zip_path))

Current folder:
c:\Projects\Text-Summarizer-project_01\Text-Summarizer-project\research

ZIP path:
c:\Projects\Text-Summarizer-project_01\Text-Summarizer-project\summarizer-data.zip

ZIP exists:
True


In [8]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_path))

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [9]:
from datasets import load_from_disk

dataset_samsum = load_from_disk("../samsum_dataset")

dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [10]:
split_lengths = [len(dataset_samsum[split])for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][1]["dialogue"])

print("\nSummary:")

print(dataset_samsum["test"][1]["summary"])

Split lengths: [14732, 819, 818]
Features: ['id', 'dialogue', 'summary']

Dialogue:
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary:
Eric and Rob are going to watch a stand-up on youtube.


In [13]:
def convert_examples_to_features(batch):
    input_encodings = tokenizer(
        batch["dialogue"],
        max_length=1024,
        truncation=True
    )

    target_encodings = tokenizer(
        text_target=batch["summary"],
        max_length=128,
        truncation=True
    )

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"]
    }

In [14]:
dataset_samsum_pt = dataset_samsum.map(
    convert_examples_to_features,
    batched=True
)

Map: 100%|██████████| 818/818 [00:00<00:00, 5457.48 examples/s]


In [16]:
dataset_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14732
})

In [17]:
# Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [19]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000000,
    gradient_accumulation_steps=16
)

In [21]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"],
    data_collator=seq2seq_data_collator
)

In [22]:
trainer.train()

c:\Projects\Text-Summarizer-project_01\Text-Summarizer-project\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
52,45.759735,2.405704


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.15s/it]


TrainOutput(global_step=52, training_loss=49.32708285405086, metrics={'train_runtime': 5324.9036, 'train_samples_per_second': 0.154, 'train_steps_per_second': 0.01, 'total_flos': 314203859361792.0, 'train_loss': 49.32708285405086, 'epoch': 1.0})

In [23]:
# Evaluation

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]



def calculate_metric_on_test_ds(dataset, metric, model, tokenizer, 
                               batch_size=16, device=device, 
                               column_text="article", 
                               column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):
        
        inputs = tokenizer(article_batch, max_length=1024,  truncation=True, 
                        padding="max_length", return_tensors="pt")
        
        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                         attention_mask=inputs["attention_mask"].to(device), 
                         length_penalty=0.8, num_beams=8, max_length=128)
        ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''
        
        # Finally, we decode the generated texts, 
        # replace the  token, and add the decoded texts with the references to the metric.
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, 
                                clean_up_tokenization_spaces=True) 
               for s in summaries]      
        
        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]
        
        
        metric.add_batch(predictions=decoded_summaries, references=target_batch)
        
    #  Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score


In [26]:
import evaluate

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

rouge_metric = evaluate.load("rouge")

In [28]:
score = calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
)

rouge_dict = {rn: score[rn] for rn in rouge_names}

pd.DataFrame(rouge_dict, index=["pegasus"])

100%|██████████| 5/5 [02:56<00:00, 35.24s/it]


,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.01995,0.0,0.019699,0.019718


In [29]:
model_pegasus.save_pretrained("pegasus-samsum-model")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.62s/it]


In [30]:
tokenizer.save_pretrained("pegasus-samsum-model")

('pegasus-samsum-model\\tokenizer_config.json',
 'pegasus-samsum-model\\tokenizer.json')

In [36]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("./pegasus-samsum-model")

In [41]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load saved model and tokenizer
model_path = "./pegasus-samsum-model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# CPU
device = torch.device("cpu")
model_pegasus = model_pegasus.to(device)

# Generation settings
gen_kwargs = {
    "length_penalty": 0.8,
    "num_beams": 8,
    "max_length": 128
}

# Get test example
sample_text = dataset_samsum["test"][0]["dialogue"]
reference = dataset_samsum["test"][0]["summary"]

# Tokenize
inputs = tokenizer(
    sample_text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

# Move inputs to CPU
inputs = {key: value.to(device) for key, value in inputs.items()}

# Generate summary
with torch.no_grad():
    summary_ids = model_pegasus.generate(
        **inputs,
        **gen_kwargs
    )

# Decode
predicted_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

print("\nModel Summary:")
print(predicted_summary)

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 2089.71it/s]


Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda: Ask Larry Amanda: He called her last time we were at the park together .<n>Hannah: I'd rather you texted him .<n>Amanda: Just text him .
